# 12: Stage 2 experiments on Kaggle 2xT4 (smoke + launch 2)
Stage 2 additions (not pre-registered; `STAGE2_PLAN.md`). Runs while HF org Jobs are blocked. **The same notebook
for every session**: it decides by itself what to do.

- **Session 1**: smoke test (NLLB translation of the multilingual prompts, a BASE A0 cell on tiny benchmark subsets,
  the 3-bit fake-quant equivalence check). Only if the smoke passes does it continue into **launch 2**: 12 runs x
  A2, A0, A3, A4q3. Each cell rebuilds the attacked weights, runs the capability benchmarks (lm-eval) and the
  multilingual / StrongREJECT generations, pushes results to the PRIVATE `<you>/aftermarket-runs`, and deletes the
  weights. Two cells at a time (one per T4). Stops starting new cells before the 12 h limit.
- **Session 2+**: resumes launch 2, skipping every cell already on the store.

Kaggle: **GPU T4 x2**, **Internet on**, secret `HF_TOKEN` attached, then **Save Version -> Save & Run All (Commit)**.
Nothing harmful is written to the notebook output: generations go only to the private store.

In [ ]:
# Session clock. Kaggle charges the whole session, so the clock starts here.
# NB_VERSION: fingerprint of this notebook's code cells, checked against the repo's copy in the next cell.
import os, sys, time
NB_VERSION = "26783e3091d2"
SESSION_START = time.time()
ON_KAGGLE = os.path.isdir("/kaggle")
print("platform:", "Kaggle" if ON_KAGGLE else "other")

In [ ]:
# (a) Clone the stage2 branch (or update it), then stop at once if these cells are an old copy.
import subprocess
REPO_URL = "https://github.com/bennytimz/aftermarket"
# The repo (and with it every generation) lives in /kaggle/temp, NOT /kaggle/working: Kaggle saves /kaggle/working
# as the notebook's output, and completions must only ever reach the private store.
if ON_KAGGLE and os.path.basename(os.getcwd()) != "aftermarket":
    os.makedirs("/kaggle/temp", exist_ok=True)
    os.chdir("/kaggle/temp")
if os.path.basename(os.getcwd()) != "aftermarket":
    if not os.path.isdir("aftermarket/.git"):
        subprocess.run(["git", "clone", "-q", "-b", "stage2", REPO_URL], check=True)
    os.chdir("aftermarket")
subprocess.run(["git", "checkout", "-q", "stage2"], check=True)
subprocess.run(["git", "pull", "-q", "--ff-only"], check=False)
print("repo at", subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())
sys.path.insert(0, "src")
from aftermarket.nbcheck import check_current
check_current(NB_VERSION, "notebooks/12_stage2_kaggle.ipynb")
print("notebook is the repo's current version:", NB_VERSION)

In [ ]:
# (b) Kaggle's own torch stays in place for training / editing / the HF fallback (Stage 1's proven install).
!pip uninstall -y -q torchao
!pip freeze | grep -iE '^(torch|torchvision|torchaudio)==' > /tmp/keep-torch.txt
!pip install -q -r requirements-kaggle.txt -c /tmp/keep-torch.txt
!pip install -q "lm-eval[ifeval]==0.4.13" "fasttext-numpy2-wheel==0.9.2" -c /tmp/keep-torch.txt
# vLLM in its OWN venv: its torch build never replaces Kaggle's. If it can't run on this GPU/driver, the driver
# falls back to the HF backend automatically (decision rule in src/aftermarket/stage2/kaggle_driver.py).
!pip install -q uv
!uv venv -q --python 3.12 /kaggle/temp/vllm-venv
!uv pip install -q --python /kaggle/temp/vllm-venv/bin/python "vllm==0.34.0" "transformers==5.22.2" "lm-eval[ifeval]==0.4.13" "pandas==3.1.1" "pyarrow==26.0.1" "pyyaml==6.0.3" "huggingface_hub"
!/kaggle/temp/vllm-venv/bin/python -c "import vllm, torch; print('venv: vllm', vllm.__version__, '| torch', torch.__version__, '| cuda', torch.version.cuda)"
import importlib.metadata as md
for p in ["torch", "transformers", "peft", "bitsandbytes", "hqq", "lm_eval", "huggingface_hub"]:
    try: print(f"{p:16s}", md.version(p))
    except md.PackageNotFoundError: print(f"{p:16s} MISSING")

In [ ]:
# (c) Hugging Face login from Kaggle secrets; the token is also passed to the worker processes.
from huggingface_hub import login, whoami
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("HF_TOKEN")  # Add-ons -> Secrets
login(token=token, add_to_git_credential=False)
os.environ["HF_TOKEN"] = token
HF_USER = whoami()["name"]
print("HF user:", HF_USER)

In [ ]:
# (d) GPUs (nvidia-smi only: the notebook process itself never imports torch)
!nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv
GPUS = "0,1"
os.environ["AFTERMARKET_TMP"] = "/kaggle/temp"
os.environ["AFTERMARKET_ADAPTERS"] = "/kaggle/temp/aftermarket_adapters"
os.environ["AFTERMARKET_VLLM_PYTHON"] = "/kaggle/temp/vllm-venv/bin/python"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.makedirs(os.environ["AFTERMARKET_ADAPTERS"], exist_ok=True)
MAX_HOURS = 11.3  # stop starting new cells after this (Kaggle's limit is 12 h)

In [ ]:
# (e) Run: smoke + launch 2 (session 1), or resume launch 2 (later sessions). Two cells at a time, one per T4.
!PYTHONPATH=src python -m aftermarket.stage2.kaggle_driver --plan auto --final --gpus {GPUS} --max-hours {MAX_HOURS} --session-start {SESSION_START}; echo $? > /tmp/driver_exit
print("driver exit code:", open("/tmp/driver_exit").read().strip())

In [ ]:
# (f) Summary of this session (cell names, return codes, minutes; no generations)
import glob, json
for f in sorted(glob.glob("results/stage2/kaggle_*.json")):
    s = json.load(open(f))
    print(f, "| plan", s.get("plan"), "| backend", s.get("backend"), "| smoke", s.get("smoke"))
    for k, v in (s.get("cells") or {}).items():
        print(f"  {k:24s} {v}")